# 04 — Explore: self vs collective framing

**Exploration only.** This notebook tries the lead question several ways to find
the honest framing. Nothing here is a final social chart — `06-viz-social` is built
only *after* the owner reviews these cuts and confirms the framing.

**Lead question:** which presidents lean on self (I/me/my) vs collective (we/us/our),
and how has it changed 1789→present?

**⭐ Apples-to-oranges guard (owner's ask):** the corpus is curated and coverage is
uneven, so per-president comparisons are only made *within a single speech type*. We
look through three lenses: the whole corpus (context only), the **State of the Union**
series (Annual Message + SOTU unified; broadest clean lens), and **Inaugural
Addresses**. Within the SOTU trend we also respect the **written-era (clerk-read,
1801–1912) vs spoken-era** break.

## ⚠️ Rendering note
This project's `.venv` is Python 3.14, where matplotlib hits a `RecursionError` even
on a plain bar + savefig (same gotcha as video-game-scores / countries-income-disparity).
So exploration uses the **shared Pillow chart factory** — which is also what social
export uses, so *what we explore here is what would ship*. Charts render inline below.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

# Shared chart kit (Pillow factory + brand palette)
SHARED = (PROJECT.parent.parent / 'shared')
sys.path.insert(0, str(SHARED))

import duckdb
from chart_factory import render_chart
from colors import c

DB = 'data/project.duckdb'
print('using', DB)

## Build the exploration aggregation tables

Derive the per-lens `chart_*` tables from `speeches_clean` so this notebook is
self-contained and reproducible (regenerate the DB → re-run this → charts still work).
We open a **writable** connection just for these builds, then close it and reconnect
**read-only** for rendering (viz never mutates the DB).

- `chart_sotu_self_share` — per-president avg self_share within SOTU (≥2 SOTUs).
- `chart_sotu_diverging` — same, centered on 0.5 (signed lean) for a diverging bar.
- `chart_inaug_self_share` — per-president self_share for inaugurals (39 presidents).
- `chart_sotu_by_decade` — avg self/collective per 1k by decade across the SOTU series.

In [ ]:
w = duckdb.connect(DB)  # writable
w.execute('DROP TABLE IF EXISTS chart_sotu_self_share')
w.execute("""
CREATE TABLE chart_sotu_self_share AS
SELECT president, COUNT(*) AS n_sotu,
       ROUND(AVG(self_share),3) AS self_share,
       printf('%.0f%%  (%d)', AVG(self_share)*100, COUNT(*)) AS label
FROM speeches_clean WHERE is_sotu_series
GROUP BY president HAVING COUNT(*)>=2
""")
w.execute('DROP TABLE IF EXISTS chart_sotu_diverging')
w.execute("""
CREATE TABLE chart_sotu_diverging AS
SELECT president, ROUND(AVG(self_share)-0.5,3) AS self_lean,
       printf('%.0f%% self', AVG(self_share)*100) AS label
FROM speeches_clean WHERE is_sotu_series
GROUP BY president HAVING COUNT(*)>=2
""")
w.execute('DROP TABLE IF EXISTS chart_inaug_self_share')
w.execute("""
CREATE TABLE chart_inaug_self_share AS
SELECT president, ROUND(AVG(self_share),3) AS self_share,
       printf('%.0f%%', AVG(self_share)*100) AS label
FROM speeches_clean WHERE speech_type='Inaugural Address'
GROUP BY president
""")
w.execute('DROP TABLE IF EXISTS chart_sotu_by_decade')
w.execute("""
CREATE TABLE chart_sotu_by_decade AS
SELECT CAST(FLOOR(year/10.0)*10 AS INTEGER) AS decade, COUNT(*) AS n,
       ROUND(AVG(self_per_1k),2) AS self_1k,
       ROUND(AVG(collective_per_1k),2) AS coll_1k,
       ROUND(AVG(self_share),3) AS self_share
FROM speeches_clean WHERE is_sotu_series GROUP BY 1 ORDER BY 1
""")
for t in ['chart_sotu_self_share','chart_sotu_diverging','chart_inaug_self_share','chart_sotu_by_decade']:
    print(t, w.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0], 'rows')
w.close()

# Reconnect read-only for all rendering below.
con = duckdb.connect(DB, read_only=True)
print('reconnected read-only')

## Coverage sanity check — why we cut by speech type

Before any comparison: confirm the whole-corpus per-president counts are wildly
uneven (a curation/coverage artifact), while the SOTU and Inaugural lenses give broad,
even presidential coverage. This is the justification for never ranking presidents on
the pooled corpus.

In [ ]:
import pandas as pd
cov = con.execute("""
  SELECT 'whole corpus' AS lens, COUNT(*) n, COUNT(DISTINCT president) presidents,
         MIN(year) yr_min, MAX(year) yr_max FROM speeches_clean
  UNION ALL SELECT 'SOTU series', COUNT(*), COUNT(DISTINCT president), MIN(year), MAX(year)
         FROM speeches_clean WHERE is_sotu_series
  UNION ALL SELECT 'Inaugural', COUNT(*), COUNT(DISTINCT president), MIN(year), MAX(year)
         FROM speeches_clean WHERE speech_type='Inaugural Address'
""").df()
print(cov.to_string(index=False))

## ⭐ The over-time story (SOTU series) — likely the strongest framing

Average self (I/me/my) vs collective (we/us/our) rate per 1,000 words, by decade,
across the State of the Union series. **This is the key exploratory finding:** the
gap is driven by *collective* language, which climbs steeply in the broadcast era —
self-reference stays comparatively flat. So the honest headline is "presidents invoke
'we' far more than they used to," **not** "presidents got more self-focused."

Reads across the written(1801–1912)/spoken break — a caption on any final version must
note that pre-1913 messages were written documents read by a clerk. `chart_sotu_by_decade`.

In [ ]:
render_chart({
    'type': 'line',
    'db': con,
    'table': 'chart_sotu_by_decade',
    'x_col': 'decade',
    'series': [
        {'col': 'coll_1k', 'label': 'collective (we/us/our)', 'color': c('teal')},
        {'col': 'self_1k', 'label': 'self (I/me/my)',         'color': c('spice')},
    ],
    'x_axis_label': 'Decade', 'y_axis_label': 'Pronouns per 1,000 words',
    'legend': True, 'label_last': False, 'markers': True, 'y_min': 0,
    'x_tick_step': 20,
    'title': 'State of the Union — self vs collective pronoun rate by decade',
    'subtitle': 'Annual Message + State of the Union, 1790–2026. Pre-1913 were written messages read by a clerk.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_self_vs_coll_by_decade',
})

## Lens 1 — State of the Union: most self-leaning presidents

Per-president average `self_share` = self / (self + collective), within the SOTU series
(≥2 SOTUs, so the average is stable). 50% = balanced. This is apples-to-apples: every
president measured on the same institutional speech. Note the early/written-era
presidents rank highest — consistent with the over-time story above. `chart_sotu_self_share`.

In [ ]:
render_chart({
    'type': 'single_ranked_bars',
    'db': con,
    'table': 'chart_sotu_self_share',
    'category_col': 'president',
    'value_col': 'self_share',
    'label_col': 'label',
    'bar_color': c('teal'),
    'title': 'State of the Union — share of first-person pronouns that are “I” not “we”',
    'subtitle': 'self ÷ (self + collective), avg across each president’s SOTUs (≥2). 50% = balanced. Label: share (n SOTUs).',
    'source': 'Miller Center (UVA) speech archive',
    'height': 1500,
    'filename': 'explore_sotu_self_share_ranked',
})

## Lens 1 (alt view) — diverging from balanced

Same SOTU data as a diverging bar centered on 50/50: right = leans “I”, left = leans
“we”. Often a cleaner social read than a ranked share. `chart_sotu_diverging`.

In [ ]:
render_chart({
    'type': 'diverging_bars',
    'db': con,
    'table': 'chart_sotu_diverging',
    'category_col': 'president',
    'value_col': 'self_lean',
    'label_col': 'label',
    'pos_color': c('spice'), 'neg_color': c('teal'),
    'zero_label': '50/50',
    'title': 'State of the Union — leaning “I” vs “we” (distance from balanced)',
    'subtitle': 'Right = more self (I/me/my); left = more collective (we/us/our). SOTU series, ≥2 speeches.',
    'source': 'Miller Center (UVA) speech archive',
    'filename': 'explore_sotu_diverging',
})

## Lens 2 — Inaugural Addresses: most self-leaning presidents

The other clean lens (39 presidents, one occasion each). Inaugurals are much more
collective than SOTUs overall — note how low the shares run — and the early presidents
(Washington especially) are the self-leaning outliers. Confirms the framing isn't an
artifact of one speech type. `chart_inaug_self_share`.

In [ ]:
render_chart({
    'type': 'single_ranked_bars',
    'db': con,
    'table': 'chart_inaug_self_share',
    'category_col': 'president',
    'value_col': 'self_share',
    'label_col': 'label',
    'bar_color': c('caramel'),
    'title': 'Inaugural Address — share of first-person pronouns that are “I” not “we”',
    'subtitle': 'self ÷ (self + collective) per inaugural (avg if two). 50% = balanced.',
    'source': 'Miller Center (UVA) speech archive',
    'height': 1500,
    'filename': 'explore_inaug_self_share_ranked',
})

## Notes for owner review (what to decide before `06-viz-social`)

- **Strongest / most honest framing:** the over-time SOTU chart. The real story is
  *collective* language rising steeply (esp. broadcast era), not presidents getting
  more self-focused — `self_share` falls over time mostly because “we” rose, not
  because “I” fell. A finding-led title would be about the rise of “we.”
- **Per-president rankings** (SOTU, inaugural) are defensible *within a lens* but skew
  toward 19th-c. presidents at the top — which is really the same time-trend restated.
  Decide if a president ranking or the trend line is the lead social chart.
- **Caveats any final chart must carry:** curated corpus; written(≤1912)-vs-spoken
  break; ghostwriting (measures the speech as delivered).
- **Open options:** cut by first-vs-second term; wartime SOTUs; add readability /
  vocabulary-richness side charts; bring in the American Presidency Project corpus to
  widen coverage (esp. press conferences, an off-the-cuff contrast).

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools. Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')